In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('./data'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import sys
print(sys.executable)
!pip install xgboost

In [ ]:
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

In [ ]:
df = pd.read_csv('./data/data_AutoReg.csv')
df.info()

In [ ]:
target_variable = 'TE_8332A.AV_0#'

# 取出目标变量列对应的原始数值（注意：target_variable需要先定义，比如 target_variable = "TE_8332A.AV_0#"）
target_column = df[target_variable].values
# 删除一些列
df = df.drop(columns = ['date', target_variable])
df.info()

In [ ]:
# 从sklearn导入MinMaxScaler，用于最小-最大归一化（把数据缩放到[0,1]区间）
from sklearn.preprocessing import MinMaxScaler

# 初始化MinMaxScaler对象
scaler = MinMaxScaler()

# 将df的所有数值列拟合并转换为缩放后的数据（每列独立缩放到[0,1]）
scaled_data = scaler.fit_transform(df.values)


In [ ]:
# 滑动窗口大小：每个样本用前12个时间步的数据
window_size = 12

# 三个列表：X_seq存3D窗口（用于LSTM等），X_lag存展平窗口（用于XGBoost等），y存目标值
X_seq, X_lag, y = [], [], []

# 从第window_size个位置开始遍历（保证每个样本都能取到前12个历史值）
for i in range(window_size, len(scaled_data)):

    # 取当前位置i之前12个时间步的窗口数据（3D形状：12 x 特征数）
    window = scaled_data[i - window_size:i]

    # 将窗口展平成一维向量（把12x特征数拍扁成一串），供XGBoost等树模型使用
    flat_window = window.flatten()  # for XGBoost

    # 存入3D窗口样本
    X_seq.append(window)

    # 存入展平窗口样本
    X_lag.append(flat_window)

    # 存入目标值（注意：用的是原始未缩放的目标列，便于评估真实误差）
    y.append(target_column[i])  # unscaled close for evaluation

# 转为numpy数组：X_seq形状为(样本数, 12, 特征数)
X_seq = np.array(X_seq)

# 转为numpy数组：X_lag形状为(样本数, 12*特征数)
X_lag = np.array(X_lag)

# 转为numpy数组：y形状为(样本数,)
y = np.array(y)


In [ ]:
X_seq_train, X_seq_test, X_lag_train, X_lag_test, y_train, y_test = train_test_split(
    X_seq, X_lag, y, test_size=0.2, shuffle=False)

In [ ]:
# xgboost训练100课决策树
model_xgb = XGBRegressor(n_estimators=100)
model_xgb.fit(X_lag_train, y_train)

In [ ]:
pred_xgb = model_xgb.predict(X_lag_test)

print('MAE of XGboost: ', mean_absolute_error(y_test,pred_xgb))
print('R2 of XGboost: ', r2_score(y_test,pred_xgb))

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(y_test, label="Actual")
plt.plot(pred_xgb, label="Predicted")
plt.title("Boiler output steam temperature prediction")
plt.xlabel("Time Step")
plt.ylabel("Boiler output steam temperature")
plt.legend()
plt.show()